In [1]:
%load_ext autoreload
%autoreload 2

import pickle
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import jax.random as jrand

from abm_event import sample_event, conditional_sample
from abm_ll import city_ll, eval_ds
from stat_utils import CityParams, EventOutcome, GeneralParams, logrange
from city_data import city_data_base, subsampled_city_data, subsample_weights


E0713 13:48:47.155078 2866357 cuda_dnn.cc:523] Loaded runtime CuDNN library: 9.5.1 but source was compiled with: 9.8.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.
E0713 13:48:47.157395 2866357 cuda_dnn.cc:523] Loaded runtime CuDNN library: 9.5.1 but source was compiled with: 9.8.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.


In [2]:
from param_distr import ParamDistr, param_config
param_distr = ParamDistr(param_config)
# params = param_distr.sample_range(jrand.PRNGKey(1))
params = GeneralParams(propensity=2e-09, walk_radius=40, arm_bias=0.01, arm_weight=1, atrisk_gathering_rate=1.02e-05)
params = jnp.array([params.propensity, params.walk_radius, params.arm_bias, params.arm_weight, params.atrisk_gathering_rate])
sample_params = param_distr.inverse_transform_sample(params)
# eval_ll = eval_params(jrand.PRNGKey(1), params, param_distr, 10000)

E0713 13:48:56.629936 2866357 cuda_dnn.cc:523] Loaded runtime CuDNN library: 9.5.1 but source was compiled with: 9.8.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.
E0713 13:48:56.635155 2866357 cuda_dnn.cc:523] Loaded runtime CuDNN library: 9.5.1 but source was compiled with: 9.8.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.


XlaRuntimeError: FAILED_PRECONDITION: DNN library initialization failed. Look at the errors above for more details.

In [52]:
sample_params = param_distr.sample_range(jrand.PRNGKey(1))
print(param_distr.transform_sample(sample_params))

[5.6823751e-08 5.1172775e+01 6.0767759e-02 7.5299603e-01 3.8855400e-05]


In [ ]:
prior = param_distr.get_uniform_prior()
log_prior = param_distr.get_log_uniform_prior()
print(prior(sample_params), log_prior(sample_params))

1.000001 9.5367386e-07


In [ ]:
from abm_mcmc import get_param_density_func
data_dict = {}
P = get_param_density_func(param_distr.get_log_uniform_prior(), jrand.PRNGKey(1), int(1e5), data_dict={})
proposal_func = param_distr.get_gaussian_proposal()
Q = param_distr.get_log_gaussian_proposal_pdf()

In [57]:
def save_data(samples, sample_eval):
    if(len(samples) % 10 != 0):
        return
    with open('data/mcmc_samples.pkl', 'wb') as f:
        pickle.dump((samples, sample_eval, data_dict), f)
    print("Saved data")

from mcmc import mcmc, mc_update #, gaussian_proposal, gaussian_proposal_pdf
samples, sample_eval = mcmc(sample_params, proposal_func, P, Q, 
                            rng=jrand.PRNGKey(0), num_iter=int(1e2), log=True, callback=save_data)

Accepted: [-7.2056885  47.552944   -1.4680734  -0.16426586 -4.6116805 ], P=-889.3767700195312
Accepted: [-7.2056885  47.552944   -1.4680734  -0.16426586 -4.6116805 ], P=-889.3767700195312
Saved data
Accepted: [-7.198284   40.92131    -1.5332687  -0.05987292 -4.560177  ], P=-886.3164672851562
Accepted: [-7.198284   40.92131    -1.5332687  -0.05987292 -4.560177  ], P=-886.3164672851562
Saved data
Accepted: [-7.198284   40.92131    -1.5332687  -0.05987292 -4.560177  ], P=-886.3164672851562
Accepted: [-7.198284   40.92131    -1.5332687  -0.05987292 -4.560177  ], P=-886.3164672851562
Saved data
Accepted: [-7.198284   40.92131    -1.5332687  -0.05987292 -4.560177  ], P=-886.3164672851562
Accepted: [-7.161357   43.45136    -1.5343646  -0.07446285 -4.6389956 ], P=-884.84521484375
Saved data
Accepted: [-7.161357   43.45136    -1.5343646  -0.07446285 -4.6389956 ], P=-884.84521484375
Accepted: [-7.161357   43.45136    -1.5343646  -0.07446285 -4.6389956 ], P=-884.84521484375
Saved data


KeyboardInterrupt: 

In [45]:
from mcmc import mcmc, mc_update #, gaussian_proposal, gaussian_proposal_pdf
samples, sample_eval = mcmc(params, proposal_func, P, Q, 
                            rng=jrand.PRNGKey(0), num_iter=int(1e2), log=True)

GeneralParams(propensity=1.9999995e-09, walk_radius=40.0, arm_bias=0.01, arm_weight=1.0, atrisk_gathering_rate=1.01999985e-05)
GeneralParams(propensity=2.1746465e-09, walk_radius=35.029545, arm_bias=0.011485768, arm_weight=1.0380658, atrisk_gathering_rate=8.959347e-06)
Accepted: [-20.030119   40.         -4.6051702   0.        -11.493123 ], P=-1352.5198974609375
GeneralParams(propensity=1.954948e-09, walk_radius=41.781967, arm_bias=0.011279838, arm_weight=1.0545423, atrisk_gathering_rate=1.4031151e-05)
Accepted: [-20.052902    41.781967    -4.4847383    0.05310681 -11.174231  ], P=-1306.16552734375
GeneralParams(propensity=1.8146162e-09, walk_radius=35.672573, arm_bias=0.01080667, arm_weight=1.0780221, atrisk_gathering_rate=1.5596854e-05)
Accepted: [-20.052902    41.781967    -4.4847383    0.05310681 -11.174231  ], P=-1306.16552734375
GeneralParams(propensity=1.9976356e-09, walk_radius=38.058327, arm_bias=0.011909851, arm_weight=1.026731, atrisk_gathering_rate=1.6046022e-05)
Accepted: 

KeyboardInterrupt: 

In [38]:
data_dict

NameError: name 'data_dict' is not defined

In [35]:
print(samples, sample_eval)

[-20.03011894  40.          -4.60517025   0.         -11.49312305
 -20.05290222  41.78196716  -4.48473835   0.05310681 -11.17423058
 -20.05290222  41.78196716  -4.48473835   0.05310681 -11.17423058] [-1357.02746582 -1308.08642578 -1308.08642578]


In [15]:
from mcmc import mcmc, mc_update, gaussian_proposal, gaussian_proposal_pdf

In [ ]:
import scipy
def bump_f(x):
    return np.exp(-x**2) + np.exp(-(x-2)**2 * 1/2)
bump_area = np.trapz(bump_f(np.linspace(-10, 10, 100)), np.linspace(-10, 10, 100))
# plt.plot(np.linspace(-10, 10, 100), bump_f(np.linspace(-10, 10, 100))/bump_area)
params = -0.9
samples, sample_eval = mcmc(params, gaussian_proposal, lambda x: np.log(bump_f(x)), 
                            lambda x, y: np.log(gaussian_proposal_pdf(x, y)), 
                            rng=np.random.default_rng(), num_iter=int(1e4), log=True)

In [ ]:

# samples, sample_eval = mcmc(params, gaussian_proposal, bump_f, 
#                             gaussian_proposal_pdf, 
#                             rng=np.random.default_rng(), num_iter=int(4e5), log=False)

In [17]:
plt.figure()
xs = np.linspace(-5, 5, 100)
plt.plot(xs, bump_f(xs)/bump_area)
plt.hist(samples[100:], bins=50, density=True)
plt.show()

NameError: name 'bump_f' is not defined

<Figure size 640x480 with 0 Axes>